In [ ]:
# !pip install pytorch_metric_learning
import pandas as pd
import numpy as np
import torch
# torch.manual_seed(1000)
# np.random.seed(1000)
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from tqdm import tqdm
from torch.optim import AdamW
from transformers import T5Tokenizer, T5ForConditionalGeneration
from pytorch_metric_learning.losses import SupConLoss

In [ ]:
## LOAD DATA

# define torch Dataset class for DataLoader
class Data(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        item = self.examples.iloc[idx]
        label = "yes" if item["label"] == 1 else "no"
        input = item['input']
        example_idx = item['example_idx']
        return input, label, example_idx

train_data_df = pd.read_csv('/path/to/T_paraphrase.csv') # load training data
train_data = DataLoader(Data(train_data_df), batch_size=64, shuffle=True) # apply torch DataLoader for batching and randomisation

In [ ]:
## LOAD MODEL

model_name = "google/flan-t5-large"
tokeniser = T5Tokenizer.from_pretrained(model_name) # load tokeniser
model_loc = "/path/to/ts1/weights" # path to ts1 model weight folder
model = T5ForConditionalGeneration.from_pretrained(model_loc,
                                                   torch_dtype=torch.bfloat16,
                                                   device_map='cuda')
optimiser = AdamW(model.parameters(), lr=5e-4)
model.gradient_checkpointing_enable() # enable gradient checkpointing due to GPU memory constraints

In [ ]:
## TS2.1

model.train()
for epoch in range(10):
    epoch_loss = 0
    for batch in tqdm(train_data): # iterate over random batches
        texts, labels, example_idx = batch # get inputs and labels from batch

        inputs = tokeniser(texts, return_tensors="pt", truncation=True, padding=True, max_length=512).to('cuda') # tokenise inputs
        labels = tokeniser(labels, return_tensors="pt", truncation=True, padding=True, max_length=2).input_ids.to("cuda") # tokenise labels

        outputs = model(**inputs, labels=labels) # forward pass
        loss = outputs.loss
        loss.backward() # backward pass
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        optimiser.zero_grad()
        epoch_loss += loss.item()

    print(f"Epoch {epoch} Loss: {epoch_loss / len(train_data_df):.4f}")

model.save_pretrained('ts2.1')

In [ ]:
mse_loss = torch.nn.MSELoss()
lamda = 0.9

def compute_mse_loss(batch_vectors, unique_input_ids):
    loss = 0
    groups = 0
    unique_ids = torch.unique(unique_input_ids) # get ids of unique inputs in random batch
    for id in unique_ids: # iterate over unique ids
        mask = (unique_input_ids == id) # get examples with current unique id
        P = batch_vectors[mask] # get representations of paraphrases in random batch
        if P.shape[0] > 1: # continue if more than one vector in batch
            v = P.mean(dim=0, keepdim=True).expand_as(P) # find mean vector of paraphrase group
            loss += mse_loss(P, v) # apply mse loss
            groups += 1 # increment groups
    if groups > 0:
        return loss / groups
    return torch.tensor(0.0).to('cuda')

for epoch in range(10):
    epoch_loss = 0
    model.train()
    for batch in tqdm(train_data):
        texts, labels, unique_input_ids = batch
        inputs = tokeniser(texts, return_tensors="pt", truncation=True, padding=True, max_length=512).to('cuda') # tokenise inputs
        tokenised_labels = tokeniser(labels, return_tensors="pt", truncation=True, padding=True, max_length=2).input_ids.to("cuda") # tokenise labels

        batch_size = inputs.input_ids.shape[0]
        decoder_input = torch.full((batch_size, 1), model.config.decoder_start_token_id, dtype=torch.long, device='cuda') # get decoder input, needed to get final decoder representation

        outputs = model(input_ids=inputs.input_ids, decoder_input_ids=decoder_input, output_hidden_states=True) # obtain model layer outputs
        decoder_output = outputs.decoder_hidden_states[-1][:, -1, :] # get last decoder representation

        logits = torch.matmul(decoder_output, model.lm_head.weight.T) # obtain logit outputs
        ce_loss = F.cross_entropy(logits, tokenised_labels[:, 0]) # find ce loss part

        unique_input_ids_tensor = torch.tensor(unique_input_ids).to('cuda')
        mse_loss_value = compute_mse_loss(decoder_output, unique_input_ids_tensor) # find mse loss part

        total_loss = (lamda * ce_loss) + ((1 - lamda) * mse_loss_value)

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        optimiser.zero_grad()
        epoch_loss += total_loss.item()

    print(f"Epoch {epoch} Loss: {epoch_loss / len(train_data_df):.4f}")

model.save_pretrained('ts2.2')

In [ ]:
## TS2.3
lamda = 0.9

def compute_cosine_loss(batch_vectors, unique_input_ids):
    loss = 0
    groups = 0
    unique_ids = torch.unique(unique_input_ids) # get ids of unique inputs in random batch
    for id in unique_ids: # iterate over unique ids
        mask = (unique_input_ids == id) # get examples with current unique id
        P = batch_vectors[mask] # get representations of paraphrases in random batch
        if P.shape[0] > 1:
            v = P.mean(dim=0, keepdim=True).expand_as(P)
            cos_sim = F.cosine_similarity(P, v, dim=1) # find cosine similarity of each representation with mean representation
            loss += (1 - cos_sim).mean() # take mean of (1 - cos_sim), to penalise low cosine similarity
            groups += 1
    if groups > 0:
        return loss / groups
    return torch.tensor(0.0).to('cuda')

## same as above for TS2.2
for epoch in range(10):
    epoch_loss = 0
    model.train()
    for batch in tqdm(train_data):
        paraphrases, labels, unique_input_ids = batch

        inputs = tokeniser(paraphrases, return_tensors="pt", truncation=True, padding=True, max_length=512).to('cuda')
        tokenised_labels = tokeniser(labels, return_tensors="pt", truncation=True, padding=True, max_length=2).input_ids.to("cuda")

        batch_size = inputs.input_ids.shape[0]
        decoder_input = torch.full((batch_size, 1), model.config.decoder_start_token_id, dtype=torch.long, device='cuda')
        outputs = model(input_ids=inputs.input_ids, decoder_input_ids=decoder_input, output_hidden_states=True)
        decoder_output = outputs.decoder_hidden_states[-1][:, -1, :]

        logits = torch.matmul(decoder_output, model.lm_head.weight.T)
        ce_loss = F.cross_entropy(logits, tokenised_labels[:, 0])

        unique_input_ids_tensor = torch.tensor(unique_input_ids).to('cuda')
        cosine_loss_value = compute_cosine_loss(decoder_output, unique_input_ids_tensor)

        total_loss = (lamda * ce_loss) + ((1 - lamda) * cosine_loss_value)
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        optimiser.zero_grad()
        epoch_loss += total_loss.item()


    print(f"Epoch {epoch} Loss: {epoch_loss / len(train_data_df):.4f}")

model.save_pretrained('ts2.3')

In [ ]:
## TS2.4

contrast_loss = SupConLoss(temperature=0.3) # initialise Pytorch Metric Learning SupConLoss instance
lamda = 0.95

for epoch in range(10):
    epoch_loss = 0
    model.train()
    for batch in tqdm(train_data):
        texts, labels, example_idx = batch

        inputs = tokeniser(texts, return_tensors="pt", truncation=True, padding=True, max_length=512).to('cuda')
        tokenised_labels = tokeniser(labels, return_tensors="pt", truncation=True, padding=True, max_length=2).input_ids.to("cuda")
        binary_labels = torch.tensor([1 if l == "yes" else 0 for l in labels]).to("cuda") # binary labels needed for contrasive loss

        batch_size = inputs.input_ids.shape[0]
        decoder_input = torch.full((batch_size, 1), model.config.decoder_start_token_id, dtype=torch.long, device='cuda')
        outputs = model(input_ids=inputs.input_ids, decoder_input_ids=decoder_input, output_hidden_states=True)
        decoder_output = outputs.decoder_hidden_states[-1][:, -1, :]

        logits = torch.matmul(decoder_output, model.lm_head.weight.T)
        ce_loss = F.cross_entropy(logits, tokenised_labels[:, 0])

        contrastive_loss = contrast_loss(decoder_output, binary_labels) # find contrastive loss part

        total_loss = (lamda * ce_loss) + ((1 - lamda) * contrastive_loss)

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        optimiser.zero_grad()
        epoch_loss += total_loss.item()

    print(f"Epoch {epoch} Loss: {epoch_loss / len(train_data_df):.4f}")

model.save_pretrained('ts2.4')